# Treino Gradient Boosting

Notebook de treino do bloco Gradient Boosting. 
A Parte I monta a infraestrutura e roda uma vez.
A Parte II treina uma área por seção, e acrescentar uma área nova custa uma célula.

Cada área é treinada nas duas codificações da variável de perdas, a que mantém as treze categorias originais e a que as agrupa em três perfis, sobre a mesma partição, os mesmos hiperparâmetros e a mesma semente. A comparação entre elas é o primeiro teste empírico da decisão sobre perdas.

**Pré-requisitos.** A partição gravada por `split_treino_teste.ipynb` e os datasets tratados precisam estar acessíveis: `particao_treino_teste.csv`, `dataset_<area>_13cat.csv`, `dataset_<area>_perfil.csv` e `limites_ativos_tratado.csv`.

**Método.** Validação por leave-one-group-out sobre as configurações de treino, com a partição lida de arquivo e nunca sorteada. Métricas calculadas uma vez sobre o conjunto das predições fora da dobra, e não como média entre dobras. Avaliação restrita a valores medidos, qualquer que seja o tratamento de ausências. Um regressor independente por componente, porque as colunas-alvo têm coberturas diferentes e o regressor multissaída exigiria descartar ensaios ou imputar rótulo. O conjunto de teste permanece reservado.

## Parte I - Infraestrutura

### 1. Configuração

Concentra os parâmetros fixos do bloco, que devem permanecer iguais nos três algoritmos do comparativo, e a localização dos arquivos. Preencher `PASTA_PROJETO` dispensa a busca automática e torna a execução reproduzível em qualquer máquina do grupo.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S4/testes_extras/testes_treinamento"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import json, os, time, warnings
import numpy as np, pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

TRATAMENTO = "mascara"     # "mascara", "imputado_treino" ou "imputado_base"
VERSOES = ("13cat", "perfil")
SEMENTE, MIN_TREINO, MIN_PREDICOES = 42, 8, 5
RUIDO_DECISAO = 0.5        # em C: abaixo disso a diferenca entre versoes e ruido
MARGEM_KPI = 10.0          # margem contratual do parceiro, em %

HIPERPARAMETROS = {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 2,
                   "min_samples_leaf": 2, "subsample": 0.8, "random_state": SEMENTE}

PREFIXOS_FEATURES = ("blower_direction_", "housing_", "board_position_", "run_case_",
                     "losses_", "perfil_perdas_", "deflector_hole_",
                     "material_coil_support_")
COLUNAS_AUXILIARES = ["run_idx"]

# Regras de nulos, iguais as do notebook de tratamento
PCT_NULOS_MAX, FAIXA_MAX, MULT_EXTREMO = 20.0, 35.0, 2.5

# Piso de erro por area (C): dispersao dos alvos entre repeticoes da mesma
# configuracao. Nenhum modelo pode ficar sistematicamente abaixo desses valores.
RUIDO_EXP = {"oven_hmi": 1.64, "ct_knobs": 2.44, "main_vent_sys": 2.71,
             "ipc_cooling": 3.62, "mainback": 4.00, "ipc_comp": 4.61,
             "coil_temps": 5.50}
AREAS = list(RUIDO_EXP)

PASTA_PROJETO = None       # ex: r"C:\Users\Inteli\Downloads\projeto\notebooks"
BASES = [".", "..", os.path.join("..", ".."), os.path.join("..", "..", ".."), "/content", os.path.join("..", "..", "..", "colabs_S3"),
         os.path.join("..", "..", "..", "colabs_S3", "preparacao_testes"), "notebooks"]
_ACHADOS = {}


def resolve(arquivo):
    """Localiza um arquivo do projeto, por caminho direto ou por varredura."""
    if arquivo in _ACHADOS and os.path.exists(_ACHADOS[arquivo]):
        return _ACHADOS[arquivo]

    for base in ([PASTA_PROJETO] if PASTA_PROJETO else []) + BASES:
        for sub in ("", "colabs_S3", os.path.join("colabs_S3", "preparacao_testes")):
            caminho = os.path.join(base, sub, arquivo)
            if os.path.exists(caminho):
                _ACHADOS[arquivo] = caminho
                return caminho

    for raiz in (os.path.abspath("."), os.path.abspath("..")):
        for pasta, subs, arquivos in os.walk(raiz):
            subs[:] = [s for s in subs if not s.startswith((".", "__", "node_modules"))]
            if arquivo in arquivos:
                _ACHADOS[arquivo] = os.path.join(pasta, arquivo)
                return _ACHADOS[arquivo]

    raise FileNotFoundError(
        f"Arquivo nao encontrado: {arquivo}. Diretorio atual: {os.path.abspath('.')}. "
        "Preencha PASTA_PROJETO com a pasta que contem os arquivos.")


print("Diretorio atual:", os.path.abspath("."))
print("Tratamento de nulos:", TRATAMENTO, "| Hiperparametros:", HIPERPARAMETROS)

### 2. Partição

A partição é lida de arquivo e nunca sorteada de novo: a variação entre sorteios foi medida em 2,25 °C, magnitude superior às diferenças que os experimentos pretendem detectar, de modo que um novo sorteio confundiria o efeito estudado com o acaso da divisão.

In [ ]:
def carrega_particao():
    """Le a particao gravada, do CSV ou, na falta dele, do JSON."""
    try:
        caminho = resolve("particao_treino_teste.csv")
        p = pd.read_csv(caminho)
        dobra = {int(r.run_idx): (None if pd.isna(r.dobra) else int(r.dobra))
                 for r in p.itertuples()}
        treino = p.loc[p["conjunto"] == "treino", "run_idx"].astype(int).tolist()
        teste = p.loc[p["conjunto"] == "teste", "run_idx"].astype(int).tolist()
        return caminho, treino, teste, dobra
    except FileNotFoundError:
        pass

    try:
        caminho = resolve("particao_treino_teste.json")
    except FileNotFoundError:
        raise FileNotFoundError("Particao nao encontrada. Rode split_treino_teste.ipynb "
                                "e disponibilize o CSV ou o JSON.")

    with open(caminho, encoding="utf-8") as f:
        reg = json.load(f)
    treino = [int(i) for i in reg["run_idx_treino"]]
    teste = [int(i) for i in reg["run_idx_teste"]]

    if "dobra_por_run_idx" in reg:
        dobra = {int(k): (None if v is None else int(v))
                 for k, v in reg["dobra_por_run_idx"].items()}
    else:
        chaves = reg.get("configuracao_por_run_idx")
        if chaves is None:
            raise KeyError("O JSON nao traz dobras nem configuracoes; use o CSV.")
        chaves = {int(k): v for k, v in chaves.items()}
        mapa = {cfg: i for i, cfg in enumerate(sorted({chaves[i] for i in treino}))}
        dobra = {i: mapa[chaves[i]] for i in treino}
        dobra.update({i: None for i in teste})
    return caminho, treino, teste, dobra


CAMINHO_PARTICAO, IDX_TREINO, IDX_TESTE, DOBRA = carrega_particao()
DOBRAS = sorted({d for d in DOBRA.values() if d is not None})

print(f"Particao: {CAMINHO_PARTICAO}")
print(f"  treino {len(IDX_TREINO)} ensaios | teste {len(IDX_TESTE)} reservados "
      f"| {len(DOBRAS)} dobras (leave-one-group-out por configuracao)")

### 3. Leitura dos dados e tratamento de ausências

As regras de ausência são as do notebook de tratamento, exclusão acima de vinte por cento de nulos com faixa superior a trinta e cinco graus, mediana na presença de valor extremo, média nos demais casos com uma diferença, o perfil da coluna é calculado apenas sobre o treino, para que o teste não contribua para o valor imputado.

No tratamento por máscara, nenhum valor é preenchido e as linhas ausentes simplesmente não entram. Em qualquer tratamento, a avaliação usa apenas valores medidos, pois pontuar contra rótulo imputado mediria a capacidade de reproduzir a imputação.

In [ ]:
def colunas_entrada(df):
    return [c for c in df.columns if c.startswith(PREFIXOS_FEATURES)]


def colunas_alvo(df):
    fora = set(colunas_entrada(df)) | set(COLUNAS_AUXILIARES)
    return [c for c in df.columns if c not in fora]


def run_idx_de(df):
    return (df["run_idx"].to_numpy().astype(int) if "run_idx" in df.columns
            else np.arange(len(df)))


def carrega_area(area, versao, tratamento=TRATAMENTO):
    """Devolve (df, medido): o dataset e a mascara de valores efetivamente medidos."""
    if tratamento == "imputado_base":
        if versao != "13cat":
            raise ValueError("arquivos snull existem apenas na versao 13cat")
        df = pd.read_csv(resolve(f"dataset_{area}_snull.csv"))
        original = pd.read_csv(resolve(f"dataset_{area}_{versao}.csv"))
        comuns = [c for c in colunas_alvo(df) if c in original.columns]
        return df, original[comuns].notna()   # sem a mascara, avaliaria rotulo fabricado
    df = pd.read_csv(resolve(f"dataset_{area}_{versao}.csv"))
    return df, df[colunas_alvo(df)].notna()


def perfila_coluna(serie):
    """Estatisticas de uma coluna, ignorando nulos."""
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    dentro = s[(s >= q1 - 1.5 * iqr) & (s <= q3 + 1.5 * iqr)]
    n = int(serie.isna().sum())
    return {"pct_nulos": 100 * n / len(serie),
            "faixa": dentro.max() - dentro.min() if len(dentro) else 0.0,
            "extremos": int(((s <= q1 - MULT_EXTREMO * iqr) |
                             (s >= q3 + MULT_EXTREMO * iqr)).sum()),
            "media": s.mean(), "mediana": s.median()}


def decide_acao(p):
    if p["pct_nulos"] > PCT_NULOS_MAX and p["faixa"] > FAIXA_MAX:
        return "excluir", f"{p['pct_nulos']:.1f}% de nulos e faixa {p['faixa']:.2f} C"
    if p["extremos"] > 0:
        return "mediana", f"{p['extremos']} valor(es) alem de {MULT_EXTREMO}x o IQR"
    return "media", "sem valores extremos"


def prepara_alvo(df, col, tratamento=TRATAMENTO):
    """Devolve (y usada no ajuste, acao). A avaliacao nunca usa esta serie."""
    y = df[col]
    if tratamento == "mascara":
        return y, "mascara"
    if tratamento == "imputado_base":
        return y, "imputado na base completa"
    p = perfila_coluna(y.iloc[IDX_TREINO])
    acao, motivo = decide_acao(p)
    if acao == "excluir":
        return None, f"excluida ({motivo})"
    return y.fillna(p[acao]), f"{acao} do treino ({p[acao]:.2f} C)"

### 4. Treino e avaliação

As dobras vêm da partição e são as mesmas para todos os componentes, áreas e versões; a máscara é aplicada depois delas, e não antes, para que componentes distintos permaneçam comparáveis entre si. Junto de cada modelo é ajustado um regressor trivial, que prevê sempre a média do treino da dobra e é a referência que dá sentido ao R² negativo. O erro sobre o próprio treino é registrado por servir ao diagnóstico de sobreajuste.

A função `verifica` roda antes de cada treino e falha explicitamente em três situações: dataset desalinhado com a partição, coluna de entrada não reconhecida e configuração presente ao mesmo tempo no treino e na validação de uma dobra.

In [ ]:
def indices_da_dobra(run, dobra, disponivel, medido):
    """(idx_treino, idx_validacao) posicionais. A validacao so admite valor medido."""
    no_treino = np.array([DOBRA.get(int(r)) is not None for r in run])
    da_dobra = np.array([DOBRA.get(int(r)) == dobra for r in run])
    return (np.where(no_treino & ~da_dobra & disponivel)[0].tolist(),
            np.where(no_treino & da_dobra & medido)[0].tolist())


def verifica(df, medido, area, versao):
    """Conferencias de integridade antes do treino."""
    cols = colunas_entrada(df)
    assert len(df) == len(DOBRA), f"{area}/{versao}: dataset e particao divergem"
    assert cols and colunas_alvo(df), f"{area}/{versao}: colunas nao reconhecidas"

    run, chave = run_idx_de(df), df[cols].astype(str).agg("|".join, axis=1)
    med = medido[medido.sum().idxmax()].to_numpy()
    vazadas = sum(bool(set(chave.iloc[tr]) & set(chave.iloc[val]))
                  for tr, val in (indices_da_dobra(run, d, med, med) for d in DOBRAS)
                  if val)
    assert vazadas == 0, f"{area}/{versao}: configuracao no treino e na validacao"


def avalia_componente(df, col, medido_col, cols_entrada, tratamento=TRATAMENTO):
    """Leave-one-group-out para um componente. Metricas sobre as predicoes acumuladas."""
    y_treino, acao = prepara_alvo(df, col, tratamento)
    if y_treino is None:
        return {"componente": col, "status": acao, "n_predicoes": 0}, None

    run, X = run_idx_de(df), df[cols_entrada]
    med = np.asarray(medido_col, dtype=bool)
    disp = y_treino.notna().to_numpy()

    reais, preditos, base, res_treino = [], [], [], []
    dobras_usadas, t0 = 0, time.perf_counter()

    for d in DOBRAS:
        idx_tr, idx_val = indices_da_dobra(run, d, disp, med)
        if len(idx_tr) < MIN_TREINO or not idx_val:
            continue
        X_tr, y_tr = X.iloc[idx_tr], y_treino.iloc[idx_tr]
        modelo = GradientBoostingRegressor(**HIPERPARAMETROS).fit(X_tr, y_tr)
        trivial = DummyRegressor(strategy="mean").fit(X_tr, y_tr)

        reais.extend(df[col].iloc[idx_val].tolist())
        preditos.extend(modelo.predict(X.iloc[idx_val]).tolist())
        base.extend(trivial.predict(X.iloc[idx_val]).tolist())
        res_treino.append(mean_absolute_error(y_tr, modelo.predict(X_tr)))
        dobras_usadas += 1

    tempo = time.perf_counter() - t0
    if len(reais) < MIN_PREDICOES:
        return {"componente": col, "status": f"poucas predicoes ({len(reais)})",
                "n_predicoes": len(reais)}, None

    reais, preditos, base = np.array(reais), np.array(preditos), np.array(base)
    return {"componente": col, "status": acao, "n_predicoes": len(reais),
            "dobras": dobras_usadas,
            "MAE": float(mean_absolute_error(reais, preditos)),
            "RMSE": float(np.sqrt(mean_squared_error(reais, preditos))),
            "R2": float(r2_score(reais, preditos)),
            "MAE_trivial": float(mean_absolute_error(reais, base)),
            "MAE_treino": float(np.mean(res_treino)),
            "faixa_alvo": float(np.ptp(reais)),
            "tempo_s": round(tempo, 2)}, {"reais": reais, "preditos": preditos}


def resumo_area(av, area):
    """Resumo de uma area: erro medio e distancia do piso de ruido."""
    mae, piso = av["MAE"].mean(), RUIDO_EXP.get(area)
    razao = mae / piso if piso else float("nan")
    return {"area": area, "componentes": len(av), "MAE": round(mae, 3),
            "RMSE": round(av["RMSE"].mean(), 3), "R2": round(av["R2"].mean(), 3),
            "MAE_trivial": round(av["MAE_trivial"].mean(), 3), "piso": piso,
            "razao_piso": round(razao, 2), "R2_negativo": int((av["R2"] < 0).sum()),
            "sem_ganho": int((av["MAE"] >= av["MAE_trivial"]).sum()),
            "tempo_s": round(av["tempo_s"].sum(), 1)}

### 5. Registro dos resultados

Cada execução é guardada em `RESULTADOS`, indexada por área e versão, com métricas e predições. É o registro que permite comparar versões, consolidar áreas e recalcular indicadores sem retreinar, e é ele que garante que uma execução nova não apague a anterior.

In [ ]:
RESULTADOS = {}


def roda(area, versao, tratamento=TRATAMENTO, rotulo=None, verbose=True):
    """Treina uma area numa versao de perdas e registra o resultado.

    rotulo permite registrar a mesma area e versao sob tratamentos de nulos
    diferentes, sem que uma execucao apague a outra.
    """
    df, medido = carrega_area(area, versao, tratamento)
    verifica(df, medido, area, versao)
    cols = colunas_entrada(df)

    linhas, predicoes = [], {}
    for col in colunas_alvo(df):
        med = medido[col] if col in medido.columns else df[col].notna()
        metricas, pred = avalia_componente(df, col, med, cols, tratamento)
        metricas.update(area=area, versao=versao, tratamento=tratamento)
        linhas.append(metricas)
        if pred is not None:
            predicoes[col] = pred

    r = pd.DataFrame(linhas)
    av = r[r["n_predicoes"] >= MIN_PREDICOES].sort_values("MAE").reset_index(drop=True)
    ig = r[r["n_predicoes"] < MIN_PREDICOES].reset_index(drop=True)

    chave = (area, rotulo or versao)
    RESULTADOS[chave] = {"avaliados": av, "ignorados": ig,
                         "resumo": resumo_area(av, area), "predicoes": predicoes,
                         "dados": df, "cols": cols, "tratamento": tratamento}
    if verbose:
        s = RESULTADOS[chave]["resumo"]
        print(f"{area} | {rotulo or versao} | {tratamento}: {len(av)} avaliados, "
              f"{len(ig)} ignorados, MAE {s['MAE']:.2f} C, R2 {s['R2']:.2f}, "
              f"{s['razao_piso']}x o piso, {s['tempo_s']:.0f} s")
    return RESULTADOS[chave]

### 6. Relatório, indicadores e comparação

`relatorio` imprime, para uma execução registrada, a tabela por componente, o resumo da área, a leitura por família, os componentes de R² negativo, o diagnóstico de sobreajuste e os indicadores do parceiro. `compara` confronta as duas versões de perdas de uma área e imprime a síntese.

Os indicadores do parceiro são reportados em três denominadores, porque a definição contratual de dez por cento ainda não está fechada e o número muda conforme a escolha. O percentual sobre a temperatura é frágil, já que a escala Celsius tem zero arbitrário; o percentual sobre o limite ativo é a leitura de engenharia; o percentual sobre a faixa mede quanto da variação existente o modelo cobre. O acerto do lado do limite não consta do requisito, mas decide a utilidade em projeto: erro menor não compensa deixar de sinalizar um estouro.

In [ ]:
import re

try:
    _lim = pd.read_csv(resolve("limites_ativos.csv"))
    LIMITE_DE = dict(zip(_lim["tc_name"], _lim["limite_num"]))
    SO_MONITORAR = set(_lim.loc[_lim["apenas_monitorar"] == True, "tc_name"])
except FileNotFoundError:
    LIMITE_DE, SO_MONITORAR = {}, set()
    print("limites_ativos_tratado.csv ausente: leituras sobre o limite ficam vazias.")


def kpis(reg, margem=MARGEM_KPI):
    """Indicadores do parceiro. Devolve (tabela por componente, resumo, criticos)."""
    linhas, criticos = [], []
    acertos = total = fn = fp = estouros = 0
    for col, p in reg["predicoes"].items():
        erro, faixa = np.abs(p["preditos"] - p["reais"]), float(np.ptp(p["reais"]))
        lim = LIMITE_DE.get(col)
        linhas.append({"componente": col, "MAE": float(erro.mean()), "limite": lim,
                       "pct_medida": float(100 * (erro / p["reais"]).mean()),
                       "pct_limite": 100 * float(erro.mean()) / lim if lim else np.nan,
                       "pct_faixa": 100 * float(erro.mean()) / faixa if faixa else np.nan})
        if lim and col not in SO_MONITORAR:
            ra, pa = p["reais"] > lim, p["preditos"] > lim
            acertos += int((ra == pa).sum()); total += len(ra)
            estouros += int(ra.sum()); fp += int((~ra & pa).sum())
            n = int((ra & ~pa).sum()); fn += n
            if n:
                criticos.append({"componente": col, "estouros": int(ra.sum()),
                                 "nao_previstos": n})

    t = pd.DataFrame(linhas)
    res = {"pct_medida": round(t["pct_medida"].mean(), 2),
           "dentro_medida": int((t["pct_medida"] <= margem).sum()),
           "pct_limite": round(t["pct_limite"].mean(), 2),
           "dentro_limite": int((t["pct_limite"] <= margem).sum()),
           "pct_faixa": round(t["pct_faixa"].mean(), 2),
           "acerto_limite": round(100 * acertos / total, 1) if total else np.nan,
           "estouros": estouros, "falsos_negativos": fn, "falsos_positivos": fp}
    return t, res, pd.DataFrame(criticos)


def relatorio(area, versao):
    """Leituras completas de uma execucao registrada."""
    reg = RESULTADOS[(area, versao)]
    av, ig, s = reg["avaliados"], reg["ignorados"], reg["resumo"]
    print("=" * 78, f"\n{area} | versao {versao} | tratamento {TRATAMENTO}\n", "=" * 78)

    print(f"\nComponentes avaliados: {len(av)} de {len(av) + len(ig)}\n")
    print(av[["componente", "n_predicoes", "MAE", "RMSE", "R2", "MAE_trivial",
              "MAE_treino", "faixa_alvo", "tempo_s"]].round(2).to_string(index=False))
    if len(ig):
        print("\nNao avaliados:")
        print(ig[["componente", "status", "n_predicoes"]].to_string(index=False))

    print("\n--- Resumo da area ---")
    print(" | ".join(f"{k} {v}" for k, v in s.items() if k != "area"))
    print(f"melhor: {av.iloc[0]['componente']} ({av.iloc[0]['MAE']:.2f} C) | "
          f"pior: {av.iloc[-1]['componente']} ({av.iloc[-1]['MAE']:.2f} C)")

    t = av.copy()
    t["familia"] = t["componente"].str.replace(r"\s*\d+$", "", regex=True).str.strip()
    t["razao_piso"] = t["MAE"] / s["piso"]
    t["razao_treino_val"] = t["MAE_treino"] / t["MAE"]

    print("\n--- Erro por familia ---")
    print(t.groupby("familia").agg(n=("componente", "count"), MAE=("MAE", "mean"),
                                   min=("MAE", "min"), max=("MAE", "max"),
                                   R2=("R2", "mean"), trivial=("MAE_trivial", "mean"))
          .sort_values("MAE").round(2).to_string())

    print(f"\n--- Piso de {s['piso']} C ---  no piso: {int((t.razao_piso < 1).sum())} | "
          f"ate 2x: {int(((t.razao_piso >= 1) & (t.razao_piso < 2)).sum())} | "
          f"acima de 2x: {int((t.razao_piso >= 2).sum())}")

    neg = t[t["R2"] < 0].sort_values("R2")
    print(f"\n--- R2 negativo: {len(neg)} de {len(t)} ---")
    if len(neg):
        print(neg[["componente", "R2", "MAE", "MAE_trivial",
                   "faixa_alvo"]].round(2).to_string(index=False))
        print("MAE abaixo do trivial: alvo de pouca variacao. Acima: as features "
              "disponiveis nao explicam o componente.")

    print(f"\n--- Treino x validacao --- razao media {t.razao_treino_val.mean():.2f} | "
          f"menor {t.razao_treino_val.min():.2f} "
          f"({t.loc[t.razao_treino_val.idxmin(), 'componente']})")

    kt, kr, criticos = kpis(reg)
    print(f"\n--- KPI de acuracidade (margem {MARGEM_KPI:.0f}%) ---")
    for rot, col in [("temperatura medida", "pct_medida"),
                     ("limite ativo", "pct_limite"), ("faixa do alvo", "pct_faixa")]:
        x = kt[col].dropna()
        print(f"  {rot:<22}{x.mean():6.1f}%  mediana {x.median():5.1f}%  "
              f"dentro: {int((x <= MARGEM_KPI).sum())} de {len(x)}")
    fora = kt[kt["pct_limite"] > MARGEM_KPI].sort_values("pct_limite", ascending=False)
    if len(fora):
        print(f"\n  acima da margem sobre o limite: {len(fora)}")
        print(fora[["componente", "MAE", "limite",
                    "pct_limite"]].round(2).to_string(index=False))

    print(f"\n--- Estouro de limite --- {kr['estouros']} estouros reais | "
          f"acerto do lado {kr['acerto_limite']}% | nao previstos {kr['falsos_negativos']} | "
          f"alarmes indevidos {kr['falsos_positivos']}")
    if len(criticos):
        print(criticos.to_string(index=False))
    return kt, kr


def compara(area, versoes=VERSOES):
    """Confronta as duas versoes de perdas de uma area e imprime a sintese."""
    a, b = (RESULTADOS[(area, v)] for v in versoes)
    print("=" * 78, f"\nComparacao {area}: {versoes[0]} x {versoes[1]}\n", "=" * 78)

    linhas = []
    for v, reg in zip(versoes, (a, b)):
        _, kr, _ = kpis(reg)
        linhas.append({**reg["resumo"], "versao": v, **kr})
    comp = pd.DataFrame(linhas)
    print(comp[["versao", "componentes", "MAE", "RMSE", "R2", "MAE_trivial",
                "razao_piso", "R2_negativo", "sem_ganho", "pct_limite",
                "acerto_limite", "falsos_negativos"]].to_string(index=False))

    d_mae = b["resumo"]["MAE"] - a["resumo"]["MAE"]
    d_r2 = b["resumo"]["R2"] - a["resumo"]["R2"]
    leitura = ("ruido" if abs(d_mae) < RUIDO_DECISAO
               else f"{versoes[1] if d_mae < 0 else versoes[0]} melhor")

    por_comp = (a["avaliados"][["componente", "MAE", "R2"]]
                .merge(b["avaliados"][["componente", "MAE", "R2"]], on="componente",
                       suffixes=(f"_{versoes[0]}", f"_{versoes[1]}")))
    por_comp["delta_MAE"] = (por_comp[f"MAE_{versoes[1]}"]
                             - por_comp[f"MAE_{versoes[0]}"])
    por_comp["leitura"] = np.where(por_comp.delta_MAE.abs() < RUIDO_DECISAO, "ruido",
                            np.where(por_comp.delta_MAE < 0, versoes[1], versoes[0]))
    por_comp = por_comp.sort_values("delta_MAE")

    print(f"\nDelta de MAE ({versoes[1]} - {versoes[0]}): {d_mae:+.2f} C | "
          f"Delta de R2: {d_r2:+.3f} | limiar {RUIDO_DECISAO} C -> {leitura}")
    print()
    print(por_comp.round(2).to_string(index=False))
    print("\nComponentes por leitura:")
    print(por_comp["leitura"].value_counts().to_string())

    cont = por_comp["leitura"].value_counts().to_dict()
    print(f"""
--- Sintese de {area} ---
Erro de {a['resumo']['MAE']:.2f} C na versao {versoes[0]} e {b['resumo']['MAE']:.2f} C na versao {versoes[1]},
contra um piso de ruido de {a['resumo']['piso']} C. A diferenca de {d_mae:+.2f} C entre as versoes e
classificada como {leitura}. Componente a componente, {cont.get(versoes[1], 0)} favorecem {versoes[1]},
{cont.get(versoes[0], 0)} favorecem {versoes[0]} e {cont.get('ruido', 0)} ficam abaixo do limiar de ruido.
Na versao {versoes[0]}, {a['resumo']['R2_negativo']} componentes tem R2 negativo e {a['resumo']['sem_ganho']} nao superam o
regressor trivial. Conclusao restrita a esta area: a decisao sobre perdas so se
sustenta apos as sete areas.""")
    return comp, por_comp


def grava(pasta="resultados_gb"):
    """Grava uma tabela por execucao e o resumo consolidado."""
    os.makedirs(pasta, exist_ok=True)
    resumos = []
    for (area, versao), reg in RESULTADOS.items():
        arq = os.path.join(pasta, f"gb_{area}_{versao}_{TRATAMENTO}.csv")
        pd.concat([reg["avaliados"], reg["ignorados"]],
                  ignore_index=True).to_csv(arq, index=False)
        _, kr, _ = kpis(reg)
        resumos.append({**reg["resumo"], "versao": versao,
                        "tratamento": reg.get("tratamento", TRATAMENTO),
                        **kr, "hiperparametros": json.dumps(HIPERPARAMETROS)})
    consolidado = pd.DataFrame(resumos)
    consolidado.to_csv(os.path.join(pasta, "gb_resumo.csv"), index=False)
    print(f"{len(resumos)} execucoes gravadas em {os.path.abspath(pasta)}")
    return consolidado

## Parte II - Áreas

Cada área ocupa uma seção com três células: treino das duas versões, relatório de cada uma e comparação com síntese. Acrescentar uma área nova é copiar a seção e trocar o nome.

### 7. IPC Comp

Maior e mais crítica das sete áreas, com 37 componentes, entre eles os dezesseis IGBTs e os oito capacitores ressonantes do módulo de potência. É a área de referência do bloco e a primeira a ser treinada.

In [ ]:
AREA = "ipc_comp"
for v in VERSOES:
    roda(AREA, v)

In [ ]:
kt13, kr13 = relatorio(AREA, "13cat")

In [ ]:
ktpf, krpf = relatorio(AREA, "perfil")

In [ ]:
comp_ipc, por_comp_ipc = compara(AREA)

### 8. Mainback

Área de faixa térmica ampla, com dez componentes que vão do indicador de painel ao enrolamento do motor de convecção. O piso de ruído é de 4,00 °C. É a área usada na construção da partição, o que não lhe dá vantagem alguma no treino: a partição depende apenas das configurações de entrada, idênticas nas sete áreas.

In [ ]:
AREA = "mainback"
for v in VERSOES:
    roda(AREA, v)

In [ ]:
relatorio(AREA, "13cat")

In [ ]:
relatorio(AREA, "perfil")

In [ ]:
comp_mainback, por_comp_mainback = compara(AREA)

### 9. Coil Temps

Área de maior faixa térmica das sete, com oito componentes divididos entre as bobinas e as placas de alumínio, e o maior piso de ruído, de 5,50 °C. Além da comparação entre as duas codificações de perdas, esta área recebe uma segunda comparação.

As variantes são o tratamento por máscara, em que a linha ausente simplesmente não entra no ajuste nem na avaliação, e o tratamento por imputação sobre o treino, em que o valor é preenchido pela mediana ou pela média conforme a regra da Seção 3. As duas usam a mesma partição e os mesmos hiperparâmetros, e a avaliação em ambas usa apenas valores medidos, de modo que a diferença observada é atribuível somente ao que entrou no ajuste.

Uma vantagem aparente da imputação precisa ser lida com cautela. Preencher o alvo com a mediana aproxima os valores ajustados da média do componente, o que reduz o erro medido sem que a capacidade preditiva tenha aumentado. O sinal de que isso ocorreu é o ganho concentrar-se nos componentes que tinham ausências, e não distribuir-se pela área.

In [ ]:
AREA = "coil_temps"
for v in VERSOES:
    roda(AREA, v)

In [ ]:
relatorio(AREA, "13cat")

In [ ]:
relatorio(AREA, "perfil")

In [ ]:
comp_coil, por_comp_coil = compara(AREA)

In [ ]:
# Variantes de tratamento dos nulos das placas de aluminio, sobre a versao 13cat.
# "mascara" corresponde a retirar as linhas ausentes; "imputado_treino" as preenche
# pela regra mediana/media ajustada sobre o conjunto de treino.
roda(AREA, "13cat", tratamento="imputado_treino", rotulo="13cat_imputado")

comp_nulos, por_comp_nulos = compara(AREA, versoes=("13cat", "13cat_imputado"))

# Onde o ganho se concentra: componentes que tinham ausencias contra os demais
df_coil, _ = carrega_area(AREA, "13cat")
com_nulos = [c for c in colunas_alvo(df_coil) if df_coil[c].isna().any()]
por_comp_nulos["tinha_nulos"] = por_comp_nulos["componente"].isin(com_nulos)

print("\nDelta de MAE por grupo de componente:")
print(por_comp_nulos.groupby("tinha_nulos")["delta_MAE"]
      .agg(componentes="count", delta_medio="mean").round(2).to_string())
print("\nGanho concentrado nos componentes que tinham ausencias indica artefato da")
print("imputacao, e nao ganho preditivo real.")

### 10. Áreas pequenas: CT Knobs, Main Vent Sys, Oven HMI e IPC Cooling

Quatro áreas de poucos componentes, agrupadas numa seção por serem rápidas de treinar e por compartilharem uma característica que muda a leitura das métricas: a faixa térmica estreita. CT Knobs tem seis componentes e piso de 2,44 °C, Main Vent Sys tem cinco e piso de 2,71 °C, Oven HMI tem quatro e o menor piso das sete áreas, 1,64 °C, e IPC Cooling tem quatro e piso de 3,62 °C.

Numa área de faixa estreita, o coeficiente de determinação castiga o modelo por uma característica do alvo e não por incapacidade preditiva: quando as temperaturas variam pouco entre configurações, sobra pouca variação a explicar, e um erro pequeno em graus pode produzir R² negativo. A leitura correta nesses casos é a comparação direta com o regressor trivial, que prevê sempre a média do treino. Um componente que não supera o trivial não está sendo explicado pelas variáveis de projeto disponíveis, e a conclusão é sobre a informação da base, não sobre o algoritmo.

A última célula da seção reúne, para as quatro áreas, o erro médio, a amplitude média dos alvos e o ganho sobre o regressor trivial, que são os três números necessários para distinguir área difícil de área com pouco a prever.

In [ ]:
AREAS_PEQUENAS = ["ct_knobs", "main_vent_sys", "oven_hmi", "ipc_cooling"]

for a in AREAS_PEQUENAS:
    for v in VERSOES:
        roda(a, v)

In [ ]:
AREA = "ct_knobs"
relatorio(AREA, "13cat")
relatorio(AREA, "perfil")
comp_ct_knobs, por_comp_ct_knobs = compara(AREA)

In [ ]:
AREA = "main_vent_sys"
relatorio(AREA, "13cat")
relatorio(AREA, "perfil")
comp_main_vent, por_comp_main_vent = compara(AREA)

In [ ]:
AREA = "oven_hmi"
relatorio(AREA, "13cat")
relatorio(AREA, "perfil")
comp_oven_hmi, por_comp_oven_hmi = compara(AREA)

In [ ]:
AREA = "ipc_cooling"
relatorio(AREA, "13cat")
relatorio(AREA, "perfil")
comp_ipc_cooling, por_comp_ipc_cooling = compara(AREA)

In [ ]:
linhas = []
for a in AREAS_PEQUENAS:
    for v in VERSOES:
        av = RESULTADOS[(a, v)]["avaliados"]
        _, kr, _ = kpis(RESULTADOS[(a, v)])
        linhas.append({
            "area": a, "versao": v, "componentes": len(av),
            "faixa_media": av["faixa_alvo"].mean(),
            "MAE": av["MAE"].mean(),
            "erro_sobre_faixa": 100 * av["MAE"].mean() / av["faixa_alvo"].mean(),
            "MAE_trivial": av["MAE_trivial"].mean(),
            "ganho_trivial": 100 * (1 - av["MAE"].mean() / av["MAE_trivial"].mean()),
            "R2": av["R2"].mean(),
            "sem_ganho": int((av["MAE"] >= av["MAE_trivial"]).sum()),
            "acerto_limite": kr["acerto_limite"],
        })

pequenas = pd.DataFrame(linhas)
print("Faixa termica e ganho sobre o regressor trivial:")
print(pequenas.round(2).to_string(index=False))

print("\nGanho negativo indica que o modelo perde para a media do treino: nessas")
print("areas a limitacao esta nas variaveis disponiveis, nao no algoritmo.")

ig = pd.concat([RESULTADOS[(a, v)]["ignorados"]
                for a in AREAS_PEQUENAS for v in VERSOES], ignore_index=True)
print(f"\nComponentes nao avaliados por cobertura insuficiente: {len(ig)}")
if len(ig):
    print(ig[["area", "versao", "componentes" if "componentes" in ig.columns
              else "componente", "status", "n_predicoes"]].to_string(index=False))

### 10. Consolidação

Reúne as execuções registradas numa tabela única e grava os arquivos. Com as sete áreas nas duas versões, é esta tabela que sustenta a decisão sobre a codificação das perdas e a comparação com os blocos de Random Forest e Extra Trees.

In [ ]:
consolidado = grava()
print()
print(consolidado[["area", "versao", "componentes", "MAE", "R2", "razao_piso",
                   "R2_negativo", "pct_limite", "acerto_limite"]].to_string(index=False))

if len(consolidado) >= 4:
    pv = consolidado.pivot(index="area", columns="versao", values="MAE")
    if set(VERSOES) <= set(pv.columns):
        pv["delta"] = pv[VERSOES[1]] - pv[VERSOES[0]]
        pv["leitura"] = np.where(pv.delta.abs() < RUIDO_DECISAO, "ruido",
                          np.where(pv.delta < 0, VERSOES[1], VERSOES[0]))
        print("\nMAE por area nas duas versoes:")
        print(pv.round(2).to_string())

### 11. Consolidação da comparação de perdas nas sete áreas

Reúne as catorze execuções numa tabela única e converte o conjunto numa recomendação sobre a codificação da variável de perdas. A pergunta é se manter as treze categorias originais, agrupá-las em três perfis, ou tratar a questão de forma diferente conforme a área.

A consolidação é feita em dois níveis, porque eles podem discordar. No nível da área, compara-se o erro médio das duas versões; no nível do componente, conta-se quantos componentes cada versão favorece. Uma diferença de área sustentada por um ou dois componentes é frágil, e só a contagem por componente revela isso. Diferenças inferiores a meio grau Celsius são tratadas como ruído em ambos os níveis.

A variante do Coil Temps com nulos imputados fica de fora desta consolidação: ela compara tratamentos de ausência, não codificações de perdas, e misturá-la à tabela confundiria as duas decisões.

In [ ]:
linhas = []
for (area, rotulo), reg in RESULTADOS.items():
    if rotulo not in VERSOES:          # exclui variantes de tratamento de nulos
        continue
    _, kr, _ = kpis(reg)
    linhas.append({**reg["resumo"], "versao": rotulo, **kr})

losses = pd.DataFrame(linhas).sort_values(["area", "versao"])
print("As sete areas nas duas versoes:")
print(losses[["area", "versao", "componentes", "MAE", "RMSE", "R2", "razao_piso",
              "R2_negativo", "sem_ganho", "pct_limite",
              "acerto_limite"]].to_string(index=False))

pv = losses.pivot(index="area", columns="versao", values="MAE")
pv["delta"] = pv[VERSOES[1]] - pv[VERSOES[0]]
pv["piso"] = [RUIDO_EXP[a] for a in pv.index]
pv["razao_piso_13cat"] = (pv[VERSOES[0]] / pv["piso"]).round(2)
pv["melhor"] = np.where(pv.delta.abs() < RUIDO_DECISAO, "ruido",
                 np.where(pv.delta < 0, VERSOES[1], VERSOES[0]))
pv = pv.sort_values("delta")

print(f"\nMAE por area, com limiar de ruido de {RUIDO_DECISAO} C:")
print(pv.round(2).to_string())
print("\nAreas por veredito:")
print(pv["melhor"].value_counts().to_string())

In [ ]:
comp_todos = []
for area in AREAS:
    if not all((area, v) in RESULTADOS for v in VERSOES):
        continue
    a = RESULTADOS[(area, VERSOES[0])]["avaliados"][["componente", "MAE"]]
    b = RESULTADOS[(area, VERSOES[1])]["avaliados"][["componente", "MAE"]]
    m = a.merge(b, on="componente", suffixes=(f"_{VERSOES[0]}", f"_{VERSOES[1]}"))
    m["area"] = area
    m["delta"] = m[f"MAE_{VERSOES[1]}"] - m[f"MAE_{VERSOES[0]}"]
    comp_todos.append(m)

comp_todos = pd.concat(comp_todos, ignore_index=True)
comp_todos["leitura"] = np.where(comp_todos.delta.abs() < RUIDO_DECISAO, "ruido",
                          np.where(comp_todos.delta < 0, VERSOES[1], VERSOES[0]))

print("Componentes por area e veredito:")
print(comp_todos.groupby("area")["leitura"].value_counts()
      .unstack(fill_value=0).to_string())
print(f"\nTotal de {len(comp_todos)} componentes:")
print(comp_todos["leitura"].value_counts().to_string())
print(f"\nDelta medio por componente: {comp_todos.delta.mean():+.3f} C | "
      f"mediana: {comp_todos.delta.median():+.3f} C")

In [ ]:
n_area = pv["melhor"].value_counts().to_dict()
n_comp = comp_todos["leitura"].value_counts().to_dict()
delta_medio = pv["delta"].mean()

if max(n_area.get(VERSOES[0], 0), n_area.get(VERSOES[1], 0)) == 0:
    rec = (f"manter as {VERSOES[0]}, por parcimonia: nenhuma area apresentou "
           "diferenca acima do limiar, e a escolha nao afeta o erro")
elif n_area.get(VERSOES[0], 0) > n_area.get(VERSOES[1], 0):
    rec = f"adotar {VERSOES[0]} em todas as areas"
elif n_area.get(VERSOES[1], 0) > n_area.get(VERSOES[0], 0):
    rec = f"adotar {VERSOES[1]} em todas as areas"
else:
    rec = "tratar por area, com justificativa individual"

print(f"""
Recomendacao sobre a codificacao das perdas.

Das {len(pv)} areas, {n_area.get(VERSOES[0], 0)} favorecem {VERSOES[0]}, {n_area.get(VERSOES[1], 0)} favorecem {VERSOES[1]} e
{n_area.get('ruido', 0)} ficam abaixo do limiar de {RUIDO_DECISAO} C. Dos {len(comp_todos)} componentes, {n_comp.get(VERSOES[0], 0)} favorecem
{VERSOES[0]}, {n_comp.get(VERSOES[1], 0)} favorecem {VERSOES[1]} e {n_comp.get('ruido', 0)} ficam no ruido. O delta medio entre
versoes e de {delta_medio:+.2f} C por area e {comp_todos.delta.mean():+.3f} C por componente.

Recomendacao: {rec}.

A versao {VERSOES[1]} usa tres colunas de perdas contra treze da {VERSOES[0]}. Quando a
diferenca de erro esta no ruido, a parcimonia decide: menos colunas esparsas
reduzem o risco de memorizacao e facilitam a leitura do modelo pela engenharia.
""")

### 12. Investigação dos componentes de desempenho fraco

Percorre os componentes cujo erro ficou muito acima do piso ou cujo coeficiente de determinação ficou negativo, e atribui a cada um uma hipótese sustentada em evidência dos dados, distinguindo o que é limitação irredutível do que é ajustável.

As hipóteses seguem quatro regras, aplicadas nesta ordem. Cobertura baixa, quando o componente tem bem menos predições que os demais da área, caso em que a métrica é pouco confiável antes de ser ruim. Features que não explicam, quando o erro iguala ou supera o do regressor trivial, situação em que nenhum ajuste de hiperparâmetro resolve, pois a informação não está na base. Alvo de pouca variação, quando o próprio regressor trivial erra menos que o piso de ruído da área, o que significa que há pouco a prever. E sobreajuste, quando o erro de treino é muito inferior ao de validação sem que as condições anteriores se apliquem, único dos quatro casos em que a solução é ajustar o modelo.

A importância das variáveis complementa o diagnóstico nas áreas problemáticas. Ela não prova causalidade, mas mostra em que o modelo está se apoiando, e uma área em que nenhuma variável concentra importância é uma área em que as variáveis disponíveis não descrevem o fenômeno.

In [ ]:
def diagnostico(area, versao):
    """Atribui uma hipotese a cada componente, com a evidencia que a sustenta."""
    reg = RESULTADOS[(area, versao)]
    t = reg["avaliados"].copy()
    piso = RUIDO_EXP[area]
    t["razao_piso"] = t.MAE / piso
    t["razao_treino_val"] = t.MAE_treino / t.MAE
    t["ganho_trivial"] = 1 - t.MAE / t.MAE_trivial
    cob_max = t.n_predicoes.max()

    def hipotese(r):
        if r.n_predicoes < 0.6 * cob_max:
            return "cobertura baixa"
        if r.MAE >= r.MAE_trivial:
            return "features nao explicam"
        if r.MAE_trivial < piso:
            return "alvo de pouca variacao"
        if r.razao_treino_val < 0.4:
            return "sobreajuste"
        return "ok"

    t["hipotese"] = t.apply(hipotese, axis=1)
    t["area"], t["versao"] = area, versao
    return t


diag = pd.concat([diagnostico(a, "13cat") for a in AREAS
                  if (a, "13cat") in RESULTADOS], ignore_index=True)

print("Hipoteses por area:")
print(diag.groupby(["area", "hipotese"]).size().unstack(fill_value=0).to_string())
print(f"\nTotal de {len(diag)} componentes:")
print(diag.hipotese.value_counts().to_string())

problemas = diag[diag.hipotese != "ok"].sort_values(["hipotese", "razao_piso"],
                                                    ascending=[True, False])
print(f"\nComponentes problematicos: {len(problemas)}")
print(problemas[["area", "componente", "MAE", "MAE_trivial", "R2", "razao_piso",
                 "razao_treino_val", "n_predicoes",
                 "hipotese"]].round(2).to_string(index=False))

In [ ]:
def grupo_feature(nome):
    for p in PREFIXOS_FEATURES:
        if nome.startswith(p):
            return p.rstrip("_")
    return "outro"


def importancias(area, versao):
    """Importancia media das variaveis, agregada por grupo de feature."""
    reg = RESULTADOS[(area, versao)]
    df, cols = reg["dados"], reg["cols"]
    no_treino = np.array([DOBRA.get(int(r)) is not None for r in run_idx_de(df)])

    imp = pd.DataFrame(index=cols)
    for col in reg["predicoes"]:
        y, _ = prepara_alvo(df, col, reg["tratamento"])
        apto = no_treino & y.notna().to_numpy()
        modelo = GradientBoostingRegressor(**HIPERPARAMETROS).fit(df[cols][apto], y[apto])
        imp[col] = modelo.feature_importances_

    grupos = pd.Series([grupo_feature(c) for c in imp.index], index=imp.index)
    return 100 * imp.groupby(grupos).sum().mean(axis=1).sort_values(ascending=False)


areas_fracas = (problemas.groupby("area").size()
                .sort_values(ascending=False).head(3).index.tolist())
print("Areas com mais componentes problematicos:", areas_fracas)

for a in areas_fracas:
    print(f"\n--- {a} | importancia media das variaveis (%) ---")
    print(importancias(a, "13cat").round(1).to_string())

In [ ]:
cont = problemas.hipotese.value_counts().to_dict()
irredutivel = cont.get("features nao explicam", 0) + cont.get("alvo de pouca variacao", 0)
ajustavel = cont.get("sobreajuste", 0)
dados = cont.get("cobertura baixa", 0)
razao_media = diag.razao_treino_val.mean()

print(f"""
Investigacao dos componentes de desempenho fraco.

Dos {len(diag)} componentes avaliados, {len(problemas)} apresentam desempenho fraco.
Destes, {irredutivel} decorrem de limitacao de informacao, {ajustavel} de sobreajuste e {dados} de
cobertura insuficiente.

A razao media entre erro de treino e de validacao e de {razao_media:.2f}. Valores muito
abaixo de 1 indicam que o modelo ajusta o treino com folga e nao transfere o
desempenho para configuracoes nao vistas.

Proximos passos, por categoria:

- Features que nao explicam ({cont.get('features nao explicam', 0)} componentes): limitacao irredutivel com as
  variaveis atuais. Levar a lista a Heitor Carneiro como pedido de variaveis
  adicionais de projeto. Nenhum ajuste de hiperparametro corrige.

- Alvo de pouca variacao ({cont.get('alvo de pouca variacao', 0)} componentes): o regressor trivial ja erra menos
  que o piso de ruido da area. Registrar como caso em que o modelo e dispensavel,
  e nao como falha.

- Sobreajuste ({ajustavel} componentes): reduzir n_estimators, aumentar min_samples_leaf
  e diminuir subsample, nesta ordem, medindo o efeito na razao treino/validacao.
  Alteracao de hiperparametro deve ser registrada e aplicada aos tres algoritmos.

- Cobertura baixa ({dados} componentes): a metrica e pouco confiavel antes de ser ruim.
  Confirmar com Heitor se a ausencia e estrutural ou falha de instrumentacao.
""")

### 13. Nota metodológica

Todos os números deste notebook são de validação, obtidos por predições fora da dobra sobre o conjunto de treinamento. O conjunto de teste permanece reservado e será usado uma única vez, ao final da seleção entre os três algoritmos, uma vez que um conjunto que influencia a escolha deixa de fornecer estimativa honesta do erro.

Três pontos precisam permanecer idênticos nos três blocos de algoritmo, sob pena de a comparação final medir configuração em vez de capacidade do modelo: a partição, lida de arquivo e nunca sorteada; as dobras, por leave-one-group-out sobre as configurações de treino; e o cálculo das métricas, feito uma vez sobre o conjunto das predições fora da dobra.